In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import pytorch_lightning as pl
from ncps.torch import CfCCell

In [13]:
batch_size = 256 #placeholder well decide the actual value later
image_h=256
image_w=384
S=8
image_feature_size=960
hidden_size=512
backbone_units=1024
ts = torch.ones(batch_size)

In [ ]:
class CoordConv(nn.Module):
   
    def __init__(self, in_channels, out_channels, kernel_size=1, padding=0):
        super().__init__()

        self.conv = nn.Conv2d(in_channels + 2, out_channels, kernel_size=kernel_size, padding=padding)

    def forward(self, x):
        batch_size, _, h, w = x.size()
        
    
        y_grid = torch.linspace(-1, 1, h, device=x.device).view(1, 1, h, 1).expand(batch_size, 1, h, w)
        x_grid = torch.linspace(-1, 1, w, device=x.device).view(1, 1, 1, w).expand(batch_size, 1, h, w)
        
        
        x_coord = torch.cat([x, y_grid, x_grid], dim=1)
        return self.conv(x_coord)

In [ ]:
class GazeLLNArch(pl.LightningModule):
    def __init__(self, image_h: int = 256, image_w: int = 384, S: int = 8, 
                 hidden_size: int = 512, backbone_units: int = 1024):
        super().__init__()
        
        self.save_hyperparameters()

        
        self.hmap_h = self.hparams.image_h // self.hparams.S
        self.hmap_w = self.hparams.image_w // self.hparams.S
        self.hmap_feature_size = self.hmap_h * self.hmap_w

        
        mobilenet = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        self.feature_extractor = mobilenet.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.image_feature_size = 960 

        
        self.coordconv = CoordConv(in_channels=1, out_channels=1, kernel_size=3, padding=1)

        
        cfc_input_size = self.image_feature_size + self.hmap_feature_size
        self.cfc_cell = CfCCell(
            input_size=cfc_input_size,
            hidden_size=self.hparams.hidden_size,
            backbone_activation="lecun_tanh",
            backbone_units=self.hparams.backbone_units,
            backbone_layers=1
        )
        
        
        self.project = nn.Linear(self.hparams.hidden_size, self.hmap_feature_size)

    def forward(self, image, prev_hmap, hx, ts):
        """
        Args:
            image: Visual stimulus tensor of shape (B, 3, H, W)
            prev_hmap: Previous fixation heatmap of shape (B, hmap_h, hmap_w) or (B, 1, hmap_h, hmap_w)
            hx: Hidden state tensor for the CfCCell
            ts: Elapsed time timespan tensor of shape (B,)
        """
        
        vis_features = self.feature_extractor(image)
        vis_features = self.pool(vis_features).flatten(1)  # Shape: (B, 960)

        # Ensure prev_hmap has a channel dimension (B, 1, hmap_h, hmap_w)
        if prev_hmap.dim() == 3:
            prev_hmap = prev_hmap.unsqueeze(1)
            
    
        hmap_coords = self.coordconv(prev_hmap).flatten(1)  # Shape: (B, hmap_h * hmap_w)
        
        
        x = torch.cat([vis_features, hmap_coords], dim=1)
        x, hx = self.cfc_cell(x, hx, ts)
        
        
        out_hmap = self.project(x)
        
        return out_hmap, hx